# Experiment 8 v2 — AI-AI Dense Communication Protocol

Diagnosis of v1: `text wins 40x on bits/byte` was measuring the wrong thing. Text sends a
very compressed description; dense sends 768 raw floats for a 10-way task that needs only
log2(10) ≈ 3.3 bits. The right question is: **what is the minimum bottleneck dimension k
where a compressed embedding matches text accuracy?**

New in v2:
1. **Bottleneck encoder sweep**: k ∈ {2, 4, 8, 16, 32, 64, 128, 256, 768}
2. **Multi-hop relay** (A→B→C): does meaning degrade faster in text or dense?
3. **Interpretability layer**: decode any dense vector → nearest words in all languages
4. **Compositionality in compressed space**: do analogy relations survive compression?

In [ ]:
!pip install sentence-transformers torch matplotlib scipy -q

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine as cos_dist
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

embedder = SentenceTransformer('LaBSE')
print('LaBSE loaded')

In [ ]:
# ── Concept vocabulary (40 concepts × 10 languages for interpretability) ─────
CONCEPTS_TEXT = [
    'water', 'fire',  'earth',    'sky',      'love',
    'fear',  'trust', 'light',    'dark',     'time',
    'life',  'death', 'eat',      'sleep',    'run',
    'give',  'take',  'speak',    'think',    'feel',
    'wind',  'stone', 'river',    'mountain', 'forest',
    'child', 'elder', 'friend',   'enemy',    'peace',
    'war',   'pain',  'joy',      'hope',     'dream',
    'build', 'break', 'find',     'lose',     'change',
]

# Multilingual vocabulary for interpretability layer
INTERP_VOCAB = {
    'en': ['water','fire','earth','sky','love','fear','trust','light','dark','time',
           'life','death','eat','sleep','run','give','take','speak','think','feel',
           'wind','stone','river','mountain','forest','child','elder','friend','enemy','peace',
           'war','pain','joy','hope','dream','build','break','find','lose','change'],
    'es': ['agua','fuego','tierra','cielo','amor','miedo','confianza','luz','oscuridad','tiempo',
           'vida','muerte','comer','dormir','correr','dar','tomar','hablar','pensar','sentir',
           'viento','piedra','río','montaña','bosque','niño','anciano','amigo','enemigo','paz',
           'guerra','dolor','alegría','esperanza','sueño','construir','romper','encontrar','perder','cambiar'],
    'zh': ['水','火','土','天空','爱','恐惧','信任','光','暗','时间',
           '生命','死亡','吃','睡觉','跑','给','拿','说话','思考','感觉',
           '风','石头','河流','山','森林','孩子','老人','朋友','敌人','和平',
           '战争','痛苦','喜悦','希望','梦想','建造','破坏','找到','失去','改变'],
    'ar': ['ماء','نار','أرض','سماء','حب','خوف','ثقة','ضوء','ظلام','وقت',
           'حياة','موت','أكل','نوم','ركض','أعطى','أخذ','تكلم','فكر','شعر',
           'ريح','حجر','نهر','جبل','غابة','طفل','مسن','صديق','عدو','سلام',
           'حرب','ألم','فرح','أمل','حلم','بنى','كسر','وجد','فقد','تغير'],
}

print(f'Concept pool: {len(CONCEPTS_TEXT)} concepts')
print('Embedding concept pool...')
CONCEPT_EMBS = embedder.encode(CONCEPTS_TEXT, normalize_embeddings=True)
print(f'Shape: {CONCEPT_EMBS.shape}')

In [ ]:
# ── Bottleneck autoencoder ────────────────────────────────────────────────────
class BottleneckEncoder(nn.Module):
    """768 → k → 768 with referential identification loss."""
    def __init__(self, input_dim=768, bottleneck_dim=8, hidden=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Linear(hidden//2, bottleneck_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden//2), nn.GELU(),
            nn.Linear(hidden//2, hidden),         nn.GELU(),
            nn.Linear(hidden, input_dim),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        out = self.decoder(z)
        return F.normalize(out, dim=-1)

    def forward(self, x):
        z   = self.encode(x)
        out = self.decode(z)
        return out, z


def train_bottleneck(k, concept_embs, n_epochs=200, n_distractors=9,
                     batch_size=64, lr=1e-3, device=DEVICE):
    """
    Train encoder for dimension k.
    Loss = reconstruction (cosine) + referential identification (cross-entropy).
    """
    X = torch.tensor(concept_embs, dtype=torch.float32).to(device)
    N = len(X)

    model = BottleneckEncoder(input_dim=768, bottleneck_dim=k).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(n_epochs):
        model.train()
        # sample mini-batches of (target, distractors) triples
        idx     = torch.randperm(N, device=device)[:batch_size]
        targets = X[idx]

        recon, z = model(targets)

        # reconstruction loss (cosine similarity maximisation)
        recon_loss = 1.0 - (recon * targets).sum(dim=-1).mean()

        # referential loss: can the compressed rep identify target among distractors?
        distractor_idx = torch.stack([
            torch.randperm(N, device=device)[:n_distractors+1]
            for _ in range(batch_size)
        ])  # (B, n_dist+1)
        # replace col 0 with target idx
        distractor_idx[:, 0] = idx
        candidates = X[distractor_idx]    # (B, n_dist+1, 768)
        # decode z to get query vector
        query = recon.unsqueeze(1)         # (B, 1, 768)
        scores = (candidates * query).sum(dim=-1)   # (B, n_dist+1)
        labels = torch.zeros(batch_size, dtype=torch.long, device=device)
        ref_loss = F.cross_entropy(scores, labels)

        loss = recon_loss + ref_loss
        opt.zero_grad()
        loss.backward()
        opt.step()

    return model


def eval_bottleneck(model, concept_embs, n_distractors=9, n_trials=500, device=DEVICE):
    model.eval()
    X   = torch.tensor(concept_embs, dtype=torch.float32).to(device)
    N   = len(X)
    rng = np.random.default_rng(42)
    correct = 0

    with torch.no_grad():
        for _ in range(n_trials):
            target_idx  = int(rng.integers(0, N))
            dist_idx    = rng.choice([i for i in range(N) if i != target_idx],
                                     n_distractors, replace=False)
            all_idx     = [target_idx] + list(dist_idx)
            all_vecs    = X[all_idx]

            recon, _    = model(X[target_idx:target_idx+1])
            sims        = (all_vecs * recon).sum(dim=-1)
            correct    += int(sims.argmax().item() == 0)

    return correct / n_trials


print('Bottleneck encoder defined.')

In [ ]:
# ── Sweep k ───────────────────────────────────────────────────────────────────
K_VALUES  = [2, 4, 8, 16, 32, 64, 128, 256, 768]
N_CONCEPT = len(CONCEPTS_TEXT)

# Text protocol baseline: send raw text description, embed, identify
def text_baseline_acc(concept_embs, n_distractors=9, n_trials=500):
    rng = np.random.default_rng(42)
    N   = len(concept_embs)
    correct = 0
    for _ in range(n_trials):
        t    = int(rng.integers(0, N))
        dist = rng.choice([i for i in range(N) if i != t], n_distractors, replace=False)
        all_idx = [t] + list(dist)
        sims = concept_embs[all_idx] @ concept_embs[t]
        correct += int(np.argmax(sims) == 0)
    return correct / n_trials

text_acc = text_baseline_acc(CONCEPT_EMBS)
print(f'Text baseline accuracy: {text_acc:.4f}')

results = []
trained_models = {}

for k in K_VALUES:
    print(f'Training k={k}...', end=' ', flush=True)
    m    = train_bottleneck(k, CONCEPT_EMBS, n_epochs=300)
    acc  = eval_bottleneck(m, CONCEPT_EMBS)
    bits = k * 16   # float16 bits
    bpb  = (acc * np.log2(N_CONCEPT)) / (bits / 8)  # bits/byte
    text_bpb = (text_acc * np.log2(N_CONCEPT)) / np.mean([len(c.encode()) for c in CONCEPTS_TEXT])
    results.append({'k': k, 'accuracy': acc, 'msg_bytes': bits//8, 'bits_per_byte': bpb})
    trained_models[k] = m
    print(f'acc={acc:.3f}  msg={bits//8}B  bpb={bpb:.5f}')

print(f'\nText baseline: acc={text_acc:.3f}  bpb={text_bpb:.5f}')

# Find k_90 and k_95
accs = [r['accuracy'] for r in results]
k_90 = next((results[i]['k'] for i, a in enumerate(accs) if a >= 0.90), None)
k_95 = next((results[i]['k'] for i, a in enumerate(accs) if a >= 0.95), None)
print(f'k_90 = {k_90}  (≥90% accuracy)  → {(k_90 or 0)*2} bytes')
print(f'k_95 = {k_95}  (≥95% accuracy)  → {(k_95 or 0)*2} bytes')

In [ ]:
# ── Plot bottleneck sweep ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ks   = [r['k']          for r in results]
accs = [r['accuracy']   for r in results]
bpbs = [r['bits_per_byte'] for r in results]
bytes_sent = [r['msg_bytes'] for r in results]

axes[0].plot(ks, accs, 'o-', color='#1976D2', linewidth=2, markersize=8, label='dense compressed')
axes[0].axhline(text_acc, color='#F57C00', linestyle='--', linewidth=2, label='text baseline')
axes[0].axhline(0.90, color='gray', linestyle=':', alpha=0.6, label='90% threshold')
axes[0].axhline(0.95, color='gray', linestyle='-.', alpha=0.6, label='95% threshold')
if k_90: axes[0].axvline(k_90, color='#4CAF50', linestyle=':', alpha=0.7)
if k_95: axes[0].axvline(k_95, color='#E91E63', linestyle=':', alpha=0.7)
axes[0].set_xscale('log', base=2)
axes[0].set_xlabel('Bottleneck dimension k', fontsize=11)
axes[0].set_ylabel('Identification accuracy', fontsize=11)
axes[0].set_title('Accuracy vs bottleneck dimension', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(bytes_sent, accs, 'o-', color='#1976D2', linewidth=2, markersize=8, label='dense')
mean_text_bytes = int(np.mean([len(c.encode()) for c in CONCEPTS_TEXT]))
axes[1].axvline(mean_text_bytes, color='#F57C00', linestyle='--', linewidth=2, label=f'text ({mean_text_bytes}B mean)')
axes[1].set_xlabel('Message size (bytes)', fontsize=11)
axes[1].set_ylabel('Identification accuracy', fontsize=11)
axes[1].set_title('Accuracy vs message size\n(crossover = where dense beats text size)', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Bottleneck Encoder Sweep — Finding minimum viable channel width', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp8_v2_bottleneck_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# ── Multi-hop relay: A → B → C ───────────────────────────────────────────────
# Each hop: receive vector, identify concept, re-encode and forward.
# Text relay: re-sends the text label of identified concept.
# Dense relay: re-encodes identified concept's compressed representation.

def multihop_dense(model, concept_embs, target_idx, n_hops=3,
                   noise_std=0.01, n_distractors=9, device=DEVICE):
    """Relay meaning through n_hops agents, injecting noise at each hop."""
    X   = torch.tensor(concept_embs, dtype=torch.float32).to(device)
    N   = len(X)
    rng = np.random.default_rng(target_idx)

    current_idx = target_idx
    for hop in range(n_hops):
        # encode current concept
        with torch.no_grad():
            recon, _ = model(X[current_idx:current_idx+1])
        # inject noise
        noisy = recon + torch.randn_like(recon) * noise_std
        noisy = F.normalize(noisy, dim=-1)
        # identify (n_distractors includes all other concepts)
        sims = (X * noisy).sum(dim=-1)
        current_idx = int(sims.argmax().item())

    return current_idx == target_idx


def multihop_text(concept_embs, target_idx, n_hops=3,
                  noise_std=0.01, n_distractors=9):
    """Text relay: re-embed text label at each hop."""
    N   = len(concept_embs)
    current_idx = target_idx
    for hop in range(n_hops):
        # 're-send' by looking up the text and re-embedding it
        query = concept_embs[current_idx]
        noise = np.random.randn(*query.shape) * noise_std
        noisy = query + noise
        noisy = noisy / (np.linalg.norm(noisy) + 1e-9)
        sims  = concept_embs @ noisy
        current_idx = int(np.argmax(sims))
    return current_idx == target_idx


N_TRIALS    = 200
HOP_RANGE   = [1, 2, 3, 4, 5]
NOISE_STDS  = [0.0, 0.05, 0.1, 0.2]

best_k = k_90 or 32
model_k = trained_models[best_k]
print(f'Using k={best_k} for multi-hop experiment')

print('\nMulti-hop accuracy (noise_std=0.05):')
print(f'{"Hops":>6} {"Dense k="+str(best_k):>14} {"Text relay":>12}')
print('-' * 36)
rng = np.random.default_rng(42)
hop_results = []
for n_hops in HOP_RANGE:
    dense_accs, text_accs = [], []
    for _ in range(N_TRIALS):
        t = int(rng.integers(0, len(CONCEPTS_TEXT)))
        dense_accs.append(multihop_dense(model_k, CONCEPT_EMBS, t, n_hops, noise_std=0.05))
        text_accs.append(multihop_text(CONCEPT_EMBS, t, n_hops, noise_std=0.05))
    da, ta = np.mean(dense_accs), np.mean(text_accs)
    hop_results.append({'hops': n_hops, 'dense': da, 'text': ta})
    print(f'{n_hops:>6} {da:>14.3f} {ta:>12.3f}')

In [ ]:
# ── Plot multi-hop ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hops  = [r['hops']  for r in hop_results]
dense = [r['dense'] for r in hop_results]
text  = [r['text']  for r in hop_results]

axes[0].plot(hops, dense, 'o-', color='#1976D2', linewidth=2, markersize=8, label=f'dense k={best_k}')
axes[0].plot(hops, text,  's-', color='#F57C00', linewidth=2, markersize=8, label='text relay')
axes[0].set_xlabel('Number of relay hops', fontsize=11)
axes[0].set_ylabel('End-to-end accuracy', fontsize=11)
axes[0].set_title('Multi-hop degradation (noise_std=0.05)\nDoes meaning degrade faster in text or dense?', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.05)

# Noise robustness at 3 hops
noise_dense, noise_text = [], []
rng2 = np.random.default_rng(99)
for noise_std in NOISE_STDS:
    da, ta = [], []
    for _ in range(N_TRIALS):
        t = int(rng2.integers(0, len(CONCEPTS_TEXT)))
        da.append(multihop_dense(model_k, CONCEPT_EMBS, t, n_hops=3, noise_std=noise_std))
        ta.append(multihop_text(CONCEPT_EMBS, t, n_hops=3, noise_std=noise_std))
    noise_dense.append(np.mean(da))
    noise_text.append(np.mean(ta))

axes[1].plot(NOISE_STDS, noise_dense, 'o-', color='#1976D2', linewidth=2, markersize=8, label=f'dense k={best_k}')
axes[1].plot(NOISE_STDS, noise_text,  's-', color='#F57C00', linewidth=2, markersize=8, label='text relay')
axes[1].set_xlabel('Channel noise σ', fontsize=11)
axes[1].set_ylabel('3-hop accuracy', fontsize=11)
axes[1].set_title('Noise robustness at 3 hops', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.suptitle('Multi-hop Relay Experiment — Dense vs Text Protocol', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp8_v2_multihop.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Interpretability layer ────────────────────────────────────────────────────
# For any compressed k-dim vector, find nearest words in multiple languages.
# This gives a human-readable 'translation' of any dense message.

print('Building multilingual interpretability index...')
interp_records = []
for lang, words in INTERP_VOCAB.items():
    embs = embedder.encode(words, normalize_embeddings=True)
    for word, emb in zip(words, embs):
        interp_records.append({'lang': lang, 'word': word, 'emb': emb})

interp_embs  = np.stack([r['emb']  for r in interp_records])
interp_words = [r['word'] for r in interp_records]
interp_langs = [r['lang'] for r in interp_records]
print(f'Index: {len(interp_words)} words × {len(INTERP_VOCAB)} languages')


def interpret_dense(compressed_vec, model, interp_embs, interp_words, interp_langs,
                    top_k=3, device=DEVICE):
    """
    Decode a compressed vector back to nearest words in each language.
    compressed_vec: numpy (k,) — the bottleneck representation
    Returns: dict of lang → [(word, sim), ...]
    """
    z    = torch.tensor(compressed_vec, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        recon = model.decode(z).cpu().numpy()[0]    # (768,)

    sims     = interp_embs @ recon
    results  = {}
    for lang in set(interp_langs):
        lang_mask  = np.array(interp_langs) == lang
        lang_sims  = sims.copy()
        lang_sims[~lang_mask] = -999
        top_idx    = np.argsort(-lang_sims)[:top_k]
        results[lang] = [(interp_words[i], float(sims[i])) for i in top_idx]
    return results

# Demo: interpret 5 concepts
print('\nInterpretability layer demo (compressed → nearest words per language):')
for concept in ['love', 'fear', 'dream', 'war', 'child']:
    idx  = CONCEPTS_TEXT.index(concept)
    X    = torch.tensor(CONCEPT_EMBS[idx:idx+1], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        _, z = model_k(X)
    z_np  = z.cpu().numpy()[0]
    interp = interpret_dense(z_np, model_k, interp_embs, interp_words, interp_langs, top_k=1)
    print(f'  {concept:<10} → ' + '  |  '.join(
        f"{lang}: {words[0][0]}({words[0][1]:.3f})"
        for lang, words in sorted(interp.items())
    ))

In [ ]:
# ── Compositionality in compressed space ─────────────────────────────────────
# Does semantic arithmetic survive compression?
# A - B + C ≈ D in the k-dim space?

def get_compressed(concept, model, concept_embs, concept_list, device=DEVICE):
    idx = concept_list.index(concept)
    X   = torch.tensor(concept_embs[idx:idx+1], dtype=torch.float32).to(device)
    with torch.no_grad():
        _, z = model(X)
    return z.cpu().numpy()[0]

all_compressed = {
    c: get_compressed(c, model_k, CONCEPT_EMBS, CONCEPTS_TEXT)
    for c in CONCEPTS_TEXT
}

ANALOGY_TESTS = [
    ('light',  'dark',   'love',   'fear'),
    ('dark',   'light',  'fear',   'love'),
    ('give',   'take',   'speak',  'think'),
    ('life',   'death',  'sleep',  'dream'),
    ('war',    'peace',  'pain',   'joy'),
    ('friend', 'enemy',  'hope',   'fear'),
    ('find',   'lose',   'build',  'break'),
    ('water',  'fire',   'river',  'forest'),
]

def analogy_compressed(a, b, c, expected_d, compressed_dict):
    if any(x not in compressed_dict for x in [a, b, c, expected_d]):
        return None
    query = compressed_dict[a] - compressed_dict[b] + compressed_dict[c]
    sims = {}
    for name, vec in compressed_dict.items():
        if name not in [a, b, c]:
            n_q = np.linalg.norm(query)
            n_v = np.linalg.norm(vec)
            sims[name] = float(np.dot(query, vec) / (n_q * n_v + 1e-9))
    ranked = sorted(sims.items(), key=lambda x: -x[1])
    rank   = next((i+1 for i, (n, _) in enumerate(ranked) if n == expected_d), None)
    return {'rank': rank, 'top3': ranked[:3], 'expected': expected_d}

print(f'Compositionality in k={best_k} compressed space:')
print(f'{"A":<10} {"B":<10} {"C":<10} {"Expected":>10} {"Rank":>6} {"Top-3"}')
print('-' * 80)

ranks = []
for a, b, c, d in ANALOGY_TESTS:
    r = analogy_compressed(a, b, c, d, all_compressed)
    if r is None:
        continue
    top3 = ', '.join(f'{n}({s:.3f})' for n, s in r['top3'])
    print(f'{a:<10} {b:<10} {c:<10} {d:>10} {str(r["rank"]):>6} {top3}')
    ranks.append(r['rank'])

if ranks:
    mrr = float(np.mean([1.0/r for r in ranks]))
    hits1 = sum(1 for r in ranks if r == 1) / len(ranks)
    hits3 = sum(1 for r in ranks if r <= 3) / len(ranks)
    print(f'\nCompressed space: MRR={mrr:.3f}  Hits@1={hits1:.2f}  Hits@3={hits3:.2f}')
    print('Compare with Exp 7 raw centroid space — does compression preserve compositionality?')